This will find all contractions for samples of a ball around manifold points
at layer 0

In [ ]:
# Import all basic functions
import onnx
import onnxruntime as ort
import ml_dtypes
import numpy as np
import h5py
import torch
import torch.utils.dlpack

# Import config
from config import load_config, InputSourceConfig

import importlib
import sys
import tqdm

DEVICE = torch.device("cuda")


In [10]:
# data generation
from generate_data.datagen import (
    sample_ambient_points,
    sample_around_points_gpu,
    repeat_points_to_match,
    get_token_ids_array,
    knn_gpu,
    estimate_local_tangent_space_p_var,
)

# model generation
from construct.construct import (
    ensure_base_model,
    ensure_subgraph_pullback,
    ensure_subgraphs,
)

# execution code
from execute.optimization_loop import optimization_loop, NewtonRaphson
from execute.io_onnx import *



In [3]:

# 1. Reload the parent module
importlib.reload(sys.modules["execute.optimization_loop"])
importlib.reload(sys.modules["generate_data.datagen"])
importlib.reload(sys.modules["construct.construct"])

<module 'construct.construct' from '/nfs-share/pa511/code_bases/new_jac/construct/construct.py'>

In [4]:
cfg = load_config("configs/smoll_embedding.toml")
force = False


# makes sure that our base model is loaded
ensure_base_model(cfg, force=force, verbose=False)
sub_graphs = ensure_subgraphs(cfg, force=force)

# Define output session
sess_options = ort.SessionOptions()

# Very Important --- will cause errors if missing
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_BASIC
sess_options.intra_op_num_threads = 8  # pick a value <= your usable core count

print(sub_graphs[0][0])
embedding_gen = ort.InferenceSession(
    str(sub_graphs[0][0]),
    sess_options=sess_options,
    providers=["CUDAExecutionProvider"],
)

# Renamed from generic input_names/output_names (original code reused those
# names for out_sess below, silently overwriting them) and switched from sets
# to ordered lists matching get_outputs()/get_inputs() order -- this is the
# same order .run(None, ...) returns outputs in, and is required for IOBinding
# to bind the correct output.
emb_outputs_meta = embedding_gen.get_outputs()
emb_inputs_meta = embedding_gen.get_inputs()
emb_output_names = [o.name for o in emb_outputs_meta]
emb_input_names = [o.name for o in emb_inputs_meta]
print(emb_input_names, emb_output_names)


Base model already exists, skipping export -> onnx_models/smoll/base_model.onnx
Extracted subgraph -> onnx_models/smoll/sub_blocks/input_ids.embedding.subgraph.forward.onnx
onnx_models/smoll/sub_blocks/input_ids.embedding.subgraph.forward.onnx
['input_ids'] ['embedding']


In [6]:
# Load config
cfg = load_config("configs/smoll.toml")

# Re-use all existing models
force = False

# makes sure that our base model is loaded
ensure_base_model(cfg, force=force, verbose=False)

force = True
# makes sure that out subgraphs are present
sub_graphs = ensure_subgraphs(cfg, force=force)

# makes sure that we have the jacobains
sub_graphs_pullbacks = ensure_subgraph_pullback(
    sub_graphs,
    force=force,
)


Base model already exists, skipping export -> onnx_models/smoll/base_model.onnx
Extracted subgraph -> onnx_models/smoll/sub_blocks/embedding.add_6.subgraph.forward.onnx
add_6
In additional outputs ('onnx::reducesum_output::7', 'add_6')


2026-09-08 13:56:32.838294908 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer ConstantSharing modified: 0 with status: OK
2026-09-08 13:56:32.838332039 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer LayerNormFusionL1 modified: 0 with status: OK
2026-09-08 13:56:32.838371468 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer CommonSubexpressionElimination modified: 0 with status: OK
2026-09-08 13:56:32.838382208 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer GeluFusionL1 modified: 0 with status: OK
2026-09-08 13:56:32.838392888 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer SimplifiedLayerNormFusion modified: 0 with status: OK
2026-09-08 13:56:32.838406868 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer FastGeluFusion modified: 0 with status: OK
2026-09-08 13:56:32.838417788 [I:onnxruntime:Default, graph_transformer.cc:15 Apply] GraphTransformer

In [7]:
out_sess = ort.InferenceSession(
    str(sub_graphs[0][0]),
    sess_options=sess_options,
    providers=["CUDAExecutionProvider"],
)

# Renamed + ordered lists -- see note in the embedding_gen cell above.
out_outputs_meta = out_sess.get_outputs()
out_inputs_meta = out_sess.get_inputs()
out_output_names = [o.name for o in out_outputs_meta]
out_input_names = [o.name for o in out_inputs_meta]

print(out_output_names, out_input_names)


['add_6'] ['embedding']


In [8]:
inputs = get_token_ids_array(
    cfg.input_source,
    cfg.seq,
)
new_input = np.repeat(
    inputs.reshape(1, *inputs.shape),
    cfg.vocab_size,
    axis=0,
)
new_input[..., -1] = np.arange(cfg.vocab_size).reshape(-1, 1)


2026-09-08 13:56:41,323 huggingface_hub.utils._http [WARNING] - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


tokenizer_config.json:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

In [ ]:
new_input[:10,1,-2]

In [ ]:
# x_out entries are now GPU-resident torch tensors (float32), not numpy arrays.
# input_ids stays host-side (bind_cpu_input, cheap: it's just token ids); the
# output is bound directly to GPU memory and retrieved via dlpack, avoiding
# the extra device->host->device round trip the original .astype(np.float32)
# on a numpy result implied.
gen = torch.Generator(device=DEVICE)
x_out = []
out_in_name = out_input_names[0]
out_out_name = out_output_names[0]
dx_list = []
dfx_list = []
for s in tqdm.tqdm(range(new_input.shape[0])):
    token_ids = new_input[s]
    x_bf16 = run_iobinding_cpu_input_gpu_output(
        embedding_gen, emb_input_names[0], token_ids, emb_output_names[0],
    )
    y0_bf16 = sample_around_points_gpu(x_bf16, n_samples=100, max_radius=1e-3, generator=gen)
    _fx = run_iobinding_gpu(out_sess, out_in_name, x_bf16, out_out_name)
    x_out.append(x_bf16)
    for i in range(y0_bf16.shape[0]):
        fy_s = run_iobinding_gpu(out_sess, out_in_name, y0_bf16[i], out_out_name)

        dx_list.append((y0_bf16[i] - x_bf16).norm(dim=-1))
        dfx_list.append((fy_s - _fx).norm(dim=-1))
    

  0%|          | 0/49152 [00:00<?, ?it/s]

100%|██████████| 49152/49152 [27:51<00:00, 29.40it/s]  


In [ ]:
import matplotlib.pyplot as plt
dx = np.concatenate([d.to(torch.float32).cpu().numpy().ravel()for d in dx_list])
dfx = np.concatenate([d.to(torch.float32).cpu().numpy().ravel() for d in dfx_list])
plt.figure(figsize=(6, 6))
plt.scatter(dx, dfx, s=4, alpha=0.3)
plt.xlabel("||y - x||")
plt.ylabel("||f(y) - f(x)||")
plt.title("||f(y) - f(x)|| vs ||y - x||")
plt.xlim(0, 0.001)
plt.ylim(0, 0.001)
plt.tight_layout()
plt.savefig("lipschitz_scatter.png", dpi=150)

In [ ]:
new_input[:100,0,:].shape

In [ ]:

x_bf16 = run_iobinding_cpu_input_gpu_output(
        embedding_gen, emb_input_names[0], new_input[:32,0,:], emb_output_names[0],
    )

In [ ]:
x_bf16.shape

In [ ]:
ans = knn_gpu(x_bf16[:,-1,:],k=1000, query_block=1,ref_block=1)

What is the relative positioning of all of these points?

From the zero ids go back to vectors and then from there figure out how is this related to the kernel of the jacobian


In [ ]:
estimate_local_tangent_space_p_var(x_bf16[:,-1,:][ans[1]].to(torch.float32))[0].shape